In [5]:
from dotenv import load_dotenv
import os 

load_dotenv()

os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')


#### Data Ingestion

In [6]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(r"C:\Users\dell\OneDrive\Desktop\LLMOPS Pipeline\data\Top Python Interview Questions.pdf")

documents = loader.load()

In [7]:
documents[1].page_content

'Python\n500 PRACTICE EXAMS QUESTIONS & ANSWERS\nWITH EXPLANATION\nTable of Contents\nChapter 1. Basics of Python Programming\nChapter 2. Python Control Structures\nChapter 3. Python Functions and Modules\nChapter 4. Python Data Structures\nChapter 5. Object Oriented Programming in Python'

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 250)

text_chunks = text_splitter.split_documents(documents)

In [9]:
text_chunks[:5]

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'C:\\Users\\dell\\OneDrive\\Desktop\\LLMOPS Pipeline\\data\\Top Python Interview Questions.pdf', 'total_pages': 259, 'page': 1, 'page_label': '2'}, page_content='Python\n500 PRACTICE EXAMS QUESTIONS & ANSWERS\nWITH EXPLANATION\nTable of Contents\nChapter 1. Basics of Python Programming\nChapter 2. Python Control Structures\nChapter 3. Python Functions and Modules\nChapter 4. Python Data Structures\nChapter 5. Object Oriented Programming in Python'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '', 'source': 'C:\\Users\\dell\\OneDrive\\Desktop\\LLMOPS Pipeline\\data\\Top Python Interview Questions.pdf', 'total_pages': 259, 'page': 2, 'page_label': '3'}, page_content='CHAPTER ONE MCQ’s\nBasics of Python Programming\nQuestion 1. In Python, a list is one of the most commonly used data\nstructures, known for its ability to store a sequence of elements. What\nhappens whe

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings_model = HuggingFaceEmbeddings(model_name = 'all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1775.84it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
vector_store = FAISS.from_documents(documents=text_chunks,embedding=embeddings_model)

In [12]:
retriever = vector_store.as_retriever()

In [13]:
query = 'What is **kwargs in python?'
docs = vector_store.similarity_search(query=query,k=5)

for idx,doc in enumerate(docs):
    print(f'Document {idx+1}')
    print('='*70)
    print('Content: ',doc.page_content)

Document 1
Content:  C. *args collects keyword arguments into a list, and **kwargs only accepts
default parameters passed by the user.
D. *args is used only to pass arguments between different modules, while
**kwargs is for static arguments within the module.
Correct Answer: B
Explanation: In Python, *args is used to pass a variable number of
positional arguments to a function, collecting them into a tuple. **kwargs is
used for passing keyword arguments, collecting them into a dictionary.
Document 2
Content:  collects additional keyword arguments as a dictionary.
C. args is used only for methods, while kwargs can be used in both
functions and methods.
D. args refers to global variables, while kwargs refers to local variables
within the function.
Correct Answer: B
Explanation: In Python, *args is used in function definitions to collect
additional positional arguments into a tuple, while **kwargs collects
additional keyword arguments into a dictionary. This mechanism allows
Document 3
Co

In [14]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""You are a helpful assistant.
    Answer to all the provided questions in a well structured format.
    Keep the answer short and crisp max 7 lines.
    If you dont know the answer just say you dont know.
    Question:{question}
    Context:{context}
    Answer:
""",input_variables=['question','context'])



In [15]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [16]:
from langchain_groq import ChatGroq

llm = ChatGroq(model_name = 'llama-3.3-70B-versatile',temperature=0.5)

In [17]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {'context':retriever,'question':RunnablePassthrough()} | prompt | llm | parser
)

In [18]:
rag_chain.invoke(input='Whar are dynamic arrays in python?')

'Dynamic arrays in Python are lists. \nThey can grow and shrink in size, \nallowing for flexible data management. \nThey support operations like indexing, \nslicing, and methods like append() and pop(). \nLists are a versatile option for managing collections of items. \nThey are the correct answer, denoted as B.'